# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kubrafathima2005-kf/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Random Forest Classifier**

I chose Random Forest because it can identify patterns from multiple content performance signals. It works well for classification and can show which features are important. I will compare it with my Week-4 rule-based baseline using the same data and evaluation metric.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split method: Grouped train-test split by client.**

I will use 80% of clients for training and 20% for testing. Pages from the same client will stay in one group. This helps prevent the model from learning client-specific patterns from both training and testing data.

In [7]:
import pandas as pd

url = "https://raw.githubusercontent.com/kubrafathima2005-kf/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_id"].nunique())

Rows: 30000
Columns: 45
Clients: 32


In [8]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))
print("Training clients:", df.iloc[train_idx]["client_id"].nunique())
print("Testing clients:", df.iloc[test_idx]["client_id"].nunique())

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Use only content and metadata features; exclude trend labels and recent-period metrics
numeric_features = [
    "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
    "word_count", "char_count"
]

categorical_features = [
    "content_type", "main_intent", "competition_level",
    "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier"
]

X = df[numeric_features + categorical_features]
y = df["is_declining_label"]

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X.iloc[train_idx], y.iloc[train_idx])

# Random Forest ranking
test_data = df.iloc[test_idx].copy()
test_data["rf_score"] = model.predict_proba(X.iloc[test_idx])[:, 1]

# Week-4 rule-based baseline
stale = test_data["freshness_tier"] == "91-180"
visible = test_data["impression_tier"].isin(["moderate", "good"])
test_data["baseline_score"] = stale.astype(int) * 2 + visible.astype(int)

# Compare Precision@50 on the same test rows
baseline_top50 = test_data.sort_values(
    ["baseline_score", "impressions_last_30d"],
    ascending=[False, False]
).head(50)

model_top50 = test_data.sort_values(
    "rf_score", ascending=False
).head(50)

baseline_p50 = baseline_top50["is_declining_label"].mean()
model_p50 = model_top50["is_declining_label"].mean()

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Random Forest"],
    "Precision@50": [baseline_p50, model_p50]
})

print(comparison)

            Method  Precision@50
0  Week-4 Baseline          0.24
1    Random Forest          0.52


I trained a Random Forest Classifier and compared it with my Week-4 rule-based baseline using the same test data and Precision@50 metric.

The Week-4 baseline achieved 0.24, while Random Forest achieved 0.52. Random Forest performed better, improving Precision@50 by 28 percentage points.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The model correctly identified 1,922 declining pages but missed 1,227 declining pages. Its accuracy was 52%.

The most important features were character count, word count, and content age. This suggests that content size and age influenced the model's predictions.

The model is useful for prioritising pages, but it can still make mistakes and should not be treated as perfectly accurate.

In [13]:
from sklearn.metrics import confusion_matrix, classification_report

predictions = model.predict(X.iloc[test_idx])
actual = y.iloc[test_idx]

print("Confusion Matrix:")
print(confusion_matrix(actual, predictions))

print("\nClassification Report:")
print(classification_report(actual, predictions))

print("\nTop 10 Feature Importances:")
rf = model.named_steps["classifier"]
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance.head(10))

Confusion Matrix:
[[1303 1711]
 [1227 1922]]

Classification Report:
              precision    recall  f1-score   support

           0       0.52      0.43      0.47      3014
           1       0.53      0.61      0.57      3149

    accuracy                           0.52      6163
   macro avg       0.52      0.52      0.52      6163
weighted avg       0.52      0.52      0.52      6163


Top 10 Feature Importances:
                           Feature  Importance
6                  num__char_count    0.202562
5                  num__word_count    0.198808
0            num__content_age_days    0.164089
3                 num__competition    0.082949
2               num__search_volume    0.080801
4                         num__cpc    0.058707
1      num__days_since_last_update    0.034877
35        cat__impression_tier_low    0.018102
36   cat__impression_tier_moderate    0.012310
11  cat__main_intent_informational    0.010937


1. No target leakage: I excluded trend_direction, trend_pct, and the target label from model features.

2. Grouped split: Training and testing used different clients, preventing client overlap.

3. Model comparison: Random Forest achieved Precision@50 of 0.52, compared with 0.24 for the Week-4 baseline.

4. Limitations: The model achieved 52% accuracy and still made incorrect predictions.

5. Future improvement: I can test more features, tune model parameters, and evaluate the model on new data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.